In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# from langchain_community.document_loaders import TextLoader,PyPDFLoader

from langchain_core.runnables import RunnableLambda,RunnableParallel
from langchain.tools import tool
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated

from langchain_community.tools import DuckDuckGoSearchRun,ShellTool
from langchain_core.tools import InjectedToolArg

from langchain_community.document_loaders import TextLoader,PyPDFLoader,Docx2txtLoader,WebBaseLoader

C:\Users\Mamatha-Win10\AppData\Local\Temp\ipykernel_14148\688021366.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun,ShellTool
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
''' Resume anaylser pdf(format), word(format), (protofilo link of a person)'''

llm= ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv('GROQ_API_KEY'),
    base_url= "https://api.groq.com/openai/v1",
    max_tokens= 2000
)


In [7]:
import tiktoken
def count_tokens(text:str,max_tokens:int=6000):
    '''Trim text so it fits correctly within the token limit of the model.'''
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    if len(tokens) <= max_tokens:
        return  text
    return encoding.encode(tokens[:max_tokens])

In [8]:
# resume -- pdf, word, portfolio link
def resume_analyser(resume: str) -> str:
    resume= resume.strip().lower()
    if resume.endswith('.pdf'):
        loader = PyPDFLoader(resume)
    elif resume.endswith('.docx'):
        loader = Docx2txtLoader(resume)
    elif resume.startswith(('http://','https://')):
        loader = WebBaseLoader(resume)
    else:
        raise ValueError("Unsupported file format. Please provide a PDF, DOCX, or a URL.")
    
    docs= loader.load()
    text= "\n".join([doc.page_content for doc in docs])
    return count_tokens(text)
    
  

In [ ]:
print(resume_analyser('Mamatha Resume word-Admin.docx'))

In [12]:
resume_anaylser1_prompt= PromptTemplate(
   template='''You are a resume analyser. You will be given a resume and you have to analyse it and give the 
    output in the following format:
    **Score**: X/100
    **Overall Match**: X%
    Keywords matched: • ...
    Missing keywords: • ...
    Readability Score: X/100
    ATS Compatibility Score: X/100
    
    2-liner summary: ...
    Skill gap analysis: • ...
    Overall improvement suggestions: • ...
    Industry specific feedback: • ...
    Job: {job_description}
    Resume: {resume_text}
    Be honest, use rubrics
    ''',
    input_variables=['job_description','resume_text']
)   

In [13]:
resume_analyser_prompt2= PromptTemplate.from_template(
    """Score resume standalone (clarity, format, ATS, skills): EXACT structure:
    1. **Score**: X/100
    2. **Strengths**: • ...
    3. **Weaknesses**: • ...
    4. **Skills Mentioned**: • ...
    5. **Recommended Skills**: • ...
    6. **Next Career Steps**: • ...
    Resume: {context}"""
)

In [14]:
parser= StrOutputParser()
def run_chain(prompt,**kwargs):
    return (prompt | llm | parser).invoke(kwargs)

In [29]:
from langchain.tools import tool

@tool
def resume_analyser_tool_job_description(resume: str) -> str:
    '''Tool to analyse resume based on job description.'''
    resume_text= resume_analyser(resume)
    return run_chain(resume_analyser_prompt1,context=resume_text)
    

In [30]:
@tool
def standalone_resume_analyser(resume: str) -> str:
    '''Tool to analyse resume standalone (clarity, format, ATS, skills).'''
    resume_text= resume_analyser(resume)
    return run_chain(resume_analyser_prompt2,context=resume_text)

In [22]:
print(standalone_resume_analyser.args)

{'resume': {'title': 'Resume', 'type': 'string'}}


In [23]:
from langchain.agents import create_agent

In [25]:
graph= create_agent(
    model=llm,
    tools=[resume_analyser_tool_job_description,standalone_resume_analyser],
    system_prompt= "You are a helpful assistant to analyse resumes."
)

In [28]:
result= graph.invoke({"messages":[{"role":"user","content":"Analyse this resume: Mamatha Resume word-Admin.docx"}]})
print(result['messages'][-1].content)

I’m happy to help analyze Mamatha’s resume. Could you please provide the resume content (you can paste the text here) or upload the `.docx` file? Once I have the resume, I can run a detailed analysis and give you feedback on format, clarity, ATS‑friendliness, and how well it matches the target job.
